# 9. Doing Things at Once — `asyncio · threading · multiprocessing`
**Python Libraries Lab · Standard Library**

_Speed up waiting with async and threads, and heavy maths with processes._

Programs are slow for two very different reasons. Sometimes they are waiting, for a website, a file or a database. That is called I O bound. Other times they are busy crunching numbers. That is CPU bound. The right speed up depends on which one you have.

**In this notebook:** You will fake checking stock at five suppliers, each taking one second. One at a time takes five seconds. With asyncio or threads, it takes about one. Then you will split a heavy calculation across processes.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Nothing to install: this is part of Python's **standard library**, so it works everywhere Python runs.

## Step 1 · The slow way: one after another
Each call waits 1 second, so five calls take about 5 seconds.

In [ ]:
import time, asyncio

def check_stock_sync(supplier):
    time.sleep(1)                    # pretend network wait
    return f"{supplier}: ok"

suppliers = ["Alpha", "Beta", "Gamma", "Delta", "Omega"]
start = time.perf_counter()
print([check_stock_sync(s) for s in suppliers])
print(f"Took {time.perf_counter() - start:.1f}s")

## Step 2 · asyncio: wait for all at once
`async def` makes a coroutine. `asyncio.gather` runs them together. In Colab you can `await` directly in a cell.

In [ ]:
async def check_stock(supplier):
    await asyncio.sleep(1)           # non-blocking wait
    return f"{supplier}: ok"

start = time.perf_counter()
results = await asyncio.gather(*(check_stock(s) for s in suppliers))
print(results)
print(f"Took {time.perf_counter() - start:.1f}s")

# In a normal .py file you would write:  asyncio.run(main())

## Step 3 · Threads: same speed-up for normal code
`ThreadPoolExecutor` runs ordinary blocking functions side by side.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(check_stock_sync, suppliers))
print(results)
print(f"Took {time.perf_counter() - start:.1f}s")

## Step 4 · Processes for heavy maths
Counting primes is pure CPU work. Processes use several cores. Speed-up depends on how many cores your machine has (Colab free usually has 2).

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor

def count_primes(limit):
    count = 0
    for n in range(2, limit):
        if all(n % d for d in range(2, int(n ** 0.5) + 1)):
            count += 1
    return count

jobs = [60_000] * 4
start = time.perf_counter(); serial = [count_primes(j) for j in jobs]; t1 = time.perf_counter() - start
start = time.perf_counter()
with ProcessPoolExecutor() as pool:
    parallel = list(pool.map(count_primes, jobs))
t2 = time.perf_counter() - start
print("CPU cores:", os.cpu_count())
print(f"One by one: {t1:.2f}s | In parallel: {t2:.2f}s | same answer: {serial == parallel}")

## Step 5 · Which one should I use?
A quick rule of thumb you can keep.

In [ ]:
guide = {
    "Many web/API calls (modern code)": "asyncio",
    "A few blocking calls in existing code": "threading / ThreadPoolExecutor",
    "Heavy number crunching in pure Python": "multiprocessing / ProcessPoolExecutor",
    "Big arrays and tables": "NumPy / pandas / Dask (they do it for you)",
}
for job, tool in guide.items():
    print(f"{job:<42} -> {tool}")

## ✍️ Your turn
Write an `async` function `double(x)` that waits 0.1s and returns `x*2`. Use `asyncio.gather` to run it for `[1, 2, 3]` and store the list in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return list(answer or []) == [2, 4, 6], f"Expected [2, 4, 6], got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
async def double(x):
    await asyncio.sleep(0.1)
    return x * 2

answer = await asyncio.gather(*(double(x) for x in [1, 2, 3]))

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Concurrency**
- **async / await**
- **gather**
- **Thread pool**
- **Process pool**
- **I/O bound vs CPU bound**

**Next:** 10. Professional Scripts (`logging · argparse · unittest`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/10_logging_argparse_unittest.ipynb)

## 💻 Run it on your own computer (optional)
1. Install Python 3.10+ from python.org (on Windows tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
# nothing to install: it is built into Python
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that just opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain `.py` file, run `!` commands in the terminal instead.